# Explore the labels

Look at the distribution of every measurement and decide where the thresholds in
`configs/labeling.yaml` should sit. A threshold picked from a histogram you have
actually looked at is defensible; a threshold copied from the build guide is a
starting point.

Run `capturecheck label data` first.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from capturecheck.config import get_config
from capturecheck.io import dataset
from capturecheck.labeling import schema

cfg = get_config()
frames = dataset.read_frame_labels()
videos = dataset.read_video_labels()
print(f"{len(videos)} captures, {len(frames)} frames")
videos.head()

## Frame measurements, with the current thresholds drawn on

In [ ]:
lines = {
    "sharpness": cfg.labeling.blur.sharp_min_laplacian_var,
    "brightness": cfg.labeling.light.dark_mean,
    "features": cfg.labeling.texture.min_features,
    "noise": cfg.labeling.noise.frame_max,
    "dup_of_prev": cfg.labeling.views.duplicate_cosine,
    "overlap_next": cfg.labeling.overlap.min_overlap,
}
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (column, threshold) in zip(axes.ravel(), lines.items()):
    values = pd.to_numeric(frames[column], errors="coerce").dropna()
    ax.hist(values, bins=50, color="#4878d0")
    ax.axvline(threshold, color="crimson", linestyle="--", label=f"threshold {threshold}")
    ax.set_title(column)
    ax.legend(fontsize=8)
fig.tight_layout()

## How often does each capture label fire?

A label that fires on almost everything, or on almost nothing, is a label whose
threshold is in the wrong place for this camera.

In [ ]:
shares = (
    videos[list(schema.VIDEO_LABEL_NAMES)].apply(pd.to_numeric, errors="coerce").mean()
)
ax = shares.sort_values().plot.barh(figsize=(7, 4), color="#6acc64")
ax.set_xlabel("share of captures flagged")
ax.axvline(0.5, color="grey", linewidth=0.8)

## Do the measurements separate the broken copies from the originals?

If a degradation does not move the measurement it is supposed to move, either
the op or the check is wrong.

In [ ]:
pairs = [
    ("blur", "blur_ratio"),
    ("exposure_jitter", "brightness_std"),
    ("noise", "noise_ratio"),
    ("subsample", "unique_views"),
    ("rotation_only", "walked_ratio"),
    ("drop_sector", "coverage_empty_bins"),
]
rows = []
clean = videos[videos["source"] != "degraded"]
for op, column in pairs:
    broken = videos[videos["degrade_type"] == op]
    if broken.empty:
        continue
    rows.append(
        {
            "op": op,
            "measurement": column,
            "clean": pd.to_numeric(clean[column], errors="coerce").mean(),
            "broken": pd.to_numeric(broken[column], errors="coerce").mean(),
            "n_broken": len(broken),
        }
    )
pd.DataFrame(rows).round(3)

## Which captures went to review, and why?

In [ ]:
queue = videos[videos["review"] == 1]
print(f"{len(queue)} of {len(videos)} captures need a person")
queue[["capture_id", "review_reason"]].head(20)